# Traductor Nasa Yuwe ↔ Español — adelanto ejecutable

Sistema **multiagente** que traduce entre nasa yuwe (NY) y español (ES).
Este notebook corre el sistema de punta a punta. **No necesita Ollama** para esta demostración.

**Cómo usarlo:**
1. Ejecuta la celda de abajo y **sube el archivo `traductor_nasayuwe.zip`** cuando lo pida.
2. Ejecuta las demás celdas en orden.

_(En Kaggle: agrega el zip como *Dataset* y ajusta la ruta de descompresión.)_

In [ ]:
# 1) Subir y descomprimir el proyecto
try:
    from google.colab import files
    subido = files.upload()   # sube traductor_nasayuwe.zip
    nombre = [n for n in subido if n.endswith('.zip')][0]
except Exception:
    nombre = 'traductor_nasayuwe.zip'   # Kaggle/local: ten el zip en el directorio

import zipfile, os
with zipfile.ZipFile(nombre) as z:
    z.extractall('.')

# entrar a la carpeta del proyecto
carpeta = 'traductor_nasayuwe'
if os.path.isdir(carpeta):
    os.chdir(carpeta)
print('Carpeta de trabajo:', os.getcwd())
print('Archivos:', sorted(f for f in os.listdir('.') if not f.startswith('__')))

## 2) Probar el sistema completo (NY → ES)
Cada frase pasa por: detector → segmentador → traductor → ensamblador → revisor.

In [ ]:
from orquestador import procesar

ejemplos = [
    'qué significa aka',
    'qué significa akh',
    'traduce ewa',
    'idx tata jĩi',
]
for p in ejemplos:
    ctx = procesar(p)
    print(f'{p!r:30s} -> {ctx.respuesta_final}')

## 3) Traducir tu propia frase
Cambia el texto y ejecuta. Muestra el resultado y la traza (quién hizo qué).

In [ ]:
def traducir(frase):
    ctx = procesar(frase)
    print('Intención:', ctx.intencion, '| Dirección:', ctx.direccion)
    print('Salida   :', ctx.respuesta_final)
    if ctx.no_resueltos:
        print('No resueltos:', ctx.no_resueltos)
    print('\nTraza:')
    for t in ctx.traza:
        print('   -', t['componente'], t['detalle'])

traducir('qué significa kiwe')

## 4) Ver el segmentador (etiquetado por token)

In [ ]:
from segmentador import segmentar
r = segmentar('¿qué significa çxhãçxha? aka ewa')
for s in r['segmentos']:
    print(f"  {s['texto']!r:14s} -> {s['idioma']:12s} ({s['metodo']})")
print('Porcentajes:', r['metadatos']['porcentajes'])

## 5) Precisión del segmentador (conjunto de prueba)

In [ ]:
import subprocess
print(subprocess.run(['python','pruebas/evaluar_segmentador.py'],
                     capture_output=True, text=True).stdout)

---
**Notas honestas:** NY→ES funciona bien (diccionario de ~4.300 palabras). ES→NY es básico en este adelanto. El Detector aquí es una heurística; el real usa Qwen/Ollama. El corpus no está validado por hablantes.